In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.7 MB/s eta 0:00:00:00:0100:01
Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


# Zero-shot classifier for Q1, Q2, Q6

In [3]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Q1

In [4]:
results = zs(prompt_150,labels_150)
cor_idx = results['labels'].index(ans_150)
ground_truth_score = results['scores'][cor_idx]
ground_truth_score

0.38442012667655945

# Q2

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embedding = model.encode([prompt_150]).astype('float32')
distances, indices = index.search(embedding,k=10)
index_list = indices[0].tolist()
print(f"Rank : {index_list.index(150) + 1}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Rank : 10


# Q3

In [6]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in index_list] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [7]:
sorted_order = np.argsort(ce_scores)[::-1]
reranked_indices = np.array(index_list)[sorted_order]
print(f"New Rank: {np.where(reranked_indices==150)[0][0]+1}")

New Rank: 1


# Q4

In [8]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
prompt_42 = train.iloc[42]["prompt"]

query_embedding = model.encode([prompt_42]).astype(np.float32)
D,I = index.search(query_embedding,k=5)

docs = [kb[i] for i in I[0]]
concatenated_docs = " ".join(docs)

text = f"Context: {concatenated_docs} Question: {prompt_42}"
tokens = tokenizer(text, truncation=False)
print(len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

216


# Q5

In [9]:
true_document = kb[150]
rag_string = f"Context: {true_document} Question: {prompt_150}"
result = zs(rag_string,candidate_labels = labels_150,multi_label=False)
gt_score = dict(zip(result["labels"], result["scores"]))[ans_150]
print(gt_score)

0.9894258379936218


# Q6

In [10]:
wrong_doc = kb[999]
adv_rag_string = f"Context: {wrong_doc} Question: {prompt_150}"
result = zs(adv_rag_string,candidate_labels = labels_150,multi_label=False)
gt_score = dict(zip(result['labels'],result['scores']))[ans_150]
print(gt_score)

0.5289487838745117


# Q7

In [11]:
hits=0
for i in range(100):
    prompt = train.iloc[i]['prompt']
    correct_text = train.iloc[i][train.iloc[i]['answer']]
    embedding = model.encode([prompt]).astype(np.float32)
    D,I = index.search(embedding,k=5)
    retrieved_docs = [kb[i] for i in I[0]]
    if any(correct_text in doc for doc in retrieved_docs):
        hits += 1
print(f"Hit %: {hits}")

Hit %: 73


# Q8

In [12]:
mec = max(ce_scores)
index_m = np.where(ce_scores == mec)[0][0]
print(ce_scores)
print(index_m)

[4.6602316 4.6602316 4.7375226 4.7375226 4.6870446 4.7525764 4.7525764
 4.7525764 4.7072477 4.758512 ]
9


In [13]:
result

{'sequence': 'Context: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics. Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.',
 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."',
  'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Loren

In [15]:
map3 = 0
for i in range(20):
    prompt = train.iloc[i]['prompt']
    embedding = model.encode([prompt]).astype(np.float32)
    D,I = index.search(embedding,k=5)
    correct_text = train.iloc[i][train.iloc[i]['answer']]
    
    docs = [kb[i] for i in I[0]]
    pairs = [[prompt,doc] for doc in docs]
    ce_scores = cross_encoder.predict(pairs)
    
    max_ce = max(ce_scores)
    max_index = np.where(ce_scores == max_ce)[0][0]
    best_document = docs[max_index]
    rag_string = f"Context: {best_document} Question: {prompt}"
    
    row = train.iloc[i]
    text_to_letter = {
        str(row["A"]): "A",
        str(row["B"]): "B",
        str(row["C"]): "C",
        str(row["D"]): "D",
        str(row["E"]): "E"
    }
    labels = [row['A'],row['B'],row['C'],row['D'],row['E']]
    results = zs(rag_string,candidate_labels = labels,multi_label=False)
    correct_ans_index = results['labels'].index(correct_text)
    
    if correct_ans_index < 3:
        map3 += (1/(1+correct_ans_index))
print(f"Map3 : {map3/20}") 

Map3 : 0.975
